<a href="https://colab.research.google.com/github/weijunlaw-lgtm/RORB/blob/main/RORB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1: Install Dependencies & Fetch Multi-Year 5m Data

In [9]:
# ==============================================================================
# CELL 1 (FIXED): PURGE CORRUPT CSVs + ROBUST 5M DATA LOADER
# ==============================================================================
!pip install -q numba plotly yfinance pandas numpy pytz

import os
import glob
import subprocess
import numpy as np
import pandas as pd
import yfinance as yf

# --- CONFIGURATION ---
DATA_SOURCE = "AUTO"            # "AUTO" tries Dukascopy Multi-Year first, then auto-falls back to YFINANCE (MNQ=F)
START_DATE  = "2025-01-01"      # Smaller 1.8-year chunk prevents Colab IP rate-limit timeouts
END_DATE    = "2026-10-15"
CSV_PATH    = "./data/nas100_m5.csv"

os.makedirs("./data", exist_ok=True)

# 1. PURGE ANY 0-BYTE OR CORRUPT CSV FILES LEFT BY FAILED RUNS
for f in glob.glob("./data/*.csv"):
    if os.path.getsize(f) < 500:
        print(f"[CLEANUP] Deleting corrupt/empty 0-byte file: {f}")
        os.remove(f)

def load_market_data(source: str, start: str, end: str, csv_path: str) -> pd.DataFrame:
    df = None

    # Attempt 1: Dukascopy Multi-Year 5m NAS100 (with throttled batching to avoid empty files)
    if source in ["AUTO", "DUKASCOPY"]:
        if not os.path.exists(csv_path) or os.path.getsize(csv_path) < 500:
            print(f"[INFO] Requesting {start} -> {end} 5m NAS100 data from Dukascopy...")
            cmd = [
                "npx", "--yes", "dukascopy-node",
                "-i", "usatechidxusd",
                "-from", start,
                "-to", end,
                "-t", "m5",
                "-f", "csv",
                "-dir", "./data",
                "-fn", "nas100_m5",
                "-df", "iso",
                "-bs", "10",    # Smaller batch size to prevent Colab IP blocks
                "-bp", "1000",  # 1s pause between batches
                "-r", "3",
                "-rp", "1000",
                "-fr"
            ]
            try:
                subprocess.run(cmd, check=True, timeout=180)
            except Exception as e:
                print(f"[WARN] Dukascopy CLI error: {e}")

        # Verify file actually contains data (> 500 bytes) before reading
        if os.path.exists(csv_path) and os.path.getsize(csv_path) > 500:
            try:
                print(f"[INFO] Loading verified dataset from {csv_path} ({os.path.getsize(csv_path):,} bytes)...")
                temp_df = pd.read_csv(csv_path)
                if len(temp_df) > 10:
                    time_col = [c for c in temp_df.columns if "time" in c.lower() or "date" in c.lower()][0]
                    temp_df[time_col] = pd.to_datetime(temp_df[time_col], utc=True)
                    df = temp_df.set_index(time_col).sort_index()
            except Exception as e:
                print(f"[WARN] CSV parse failed ({e}). Removing bad file...")
                if os.path.exists(csv_path):
                    os.remove(csv_path)
        else:
            if os.path.exists(csv_path):
                os.remove(csv_path)
            print("[WARN] Dukascopy returned an empty file (Colab IP rate-limited). Switching to CME MNQ=F...")

    # Attempt 2: Automatic Fallback to Yahoo Finance CME Micro Nasdaq Futures (MNQ=F)
    if df is None or len(df) == 0:
        print("[INFO] Fetching 60 days of 5m CME Micro Nasdaq Futures (MNQ=F) via Yahoo Finance...")
        df = yf.download("MNQ=F", period="60d", interval="5m", progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.index = pd.to_datetime(df.index, utc=True)
        # Cache valid data to CSV so Cell 3 always has a valid local copy
        df.to_csv(csv_path)

    # Standardize columns and convert timezone to America/New_York
    df.columns = [str(c).lower() for c in df.columns]
    df = df[["open", "high", "low", "close"]].dropna()
    df = df[~df.index.duplicated(keep="first")]
    df.index = df.index.tz_convert("America/New_York")

    print(f"[SUCCESS] Loaded {len(df):,} 5m bars | Range: {df.index[0]} -> {df.index[-1]}")
    return df

df_5m = load_market_data(DATA_SOURCE, START_DATE, END_DATE, CSV_PATH)
df_5m.tail()

[CLEANUP] Deleting corrupt/empty 0-byte file: ./data/nas100_m5.csv
[INFO] Requesting 2025-01-01 -> 2026-10-15 5m NAS100 data from Dukascopy...
[WARN] Dukascopy CLI error: Command '['npx', '--yes', 'dukascopy-node', '-i', 'usatechidxusd', '-from', '2025-01-01', '-to', '2026-10-15', '-t', 'm5', '-f', 'csv', '-dir', './data', '-fn', 'nas100_m5', '-df', 'iso', '-bs', '10', '-bp', '1000', '-r', '3', '-rp', '1000', '-fr']' timed out after 180 seconds
[WARN] Dukascopy returned an empty file (Colab IP rate-limited). Switching to CME MNQ=F...
[INFO] Fetching 60 days of 5m CME Micro Nasdaq Futures (MNQ=F) via Yahoo Finance...


/tmp/ipykernel_732/1900009563.py:76: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("MNQ=F", period="60d", interval="5m", progress=False)


[SUCCESS] Loaded 13,448 5m bars | Range: 2026-08-07 00:05:00-04:00 -> 2026-10-16 04:50:00-04:00


,open,high,low,close
Datetime,,,,
2026-10-16 04:30:00-04:00,30182.25,30186.5,30152.50,30158.25
2026-10-16 04:35:00-04:00,30158.00,30168.5,30151.75,30164.00
2026-10-16 04:40:00-04:00,30164.00,30185.0,30162.75,30179.75
2026-10-16 04:45:00-04:00,30179.50,30185.0,30167.00,30173.50
2026-10-16 04:50:00-04:00,30173.50,30180.5,30165.50,30169.50


## Cell 2: Numba JIT RORB State Machine Engine

In [6]:
# ==============================================================================
# CELL 2: NUMBA JIT-COMPILED RORB STATE MACHINE ENGINE
# ==============================================================================
from numba import njit

@njit
def round_tick(val: float, tick_size: float) -> float:
    return np.round(val / tick_size) * tick_size

@njit
def calc_position_size(sl_dist: float, risk_usd: float, point_value: float, tick_size: float, is_futures: bool) -> float:
    safe_dist = max(sl_dist, tick_size)
    risk_per_unit = safe_dist * point_value
    raw_qty = risk_usd / risk_per_unit
    if is_futures:
        return max(1.0, np.round(raw_qty))
    return raw_qty

@njit
def rorb_state_machine_kernel(
    open_arr: np.ndarray,
    high_arr: np.ndarray,
    low_arr: np.ndarray,
    close_arr: np.ndarray,
    day_id_arr: np.ndarray,
    min_of_day_arr: np.ndarray,
    risk_usd: float,
    point_value: float,
    tick_size: float,
    comm_per_unit: float,
    is_futures: bool,
    fib_lvl: float,
    t1_min_rr: float,
    t1_max_rr: float,
    t2_fix_rr: float
):
    n = len(close_arr)
    max_trades = n // 5

    # Pre-allocated Trade Log Arrays
    tr_entry_idx = np.zeros(max_trades, dtype=np.int64)
    tr_exit_idx  = np.zeros(max_trades, dtype=np.int64)
    tr_leg       = np.zeros(max_trades, dtype=np.int64)   # 1 = Trade 1 (0.618 Limit), 2 = Trade 2 (2R Reversal)
    tr_dir       = np.zeros(max_trades, dtype=np.int64)   # 1 = Long, -1 = Short
    tr_qty       = np.zeros(max_trades, dtype=np.float64)
    tr_entry_px  = np.zeros(max_trades, dtype=np.float64)
    tr_exit_px   = np.zeros(max_trades, dtype=np.float64)
    tr_sl_px     = np.zeros(max_trades, dtype=np.float64)
    tr_tp_px     = np.zeros(max_trades, dtype=np.float64)
    tr_pnl_usd   = np.zeros(max_trades, dtype=np.float64)
    tr_r_mult    = np.zeros(max_trades, dtype=np.float64)
    t_count      = 0

    # State Machine Variables
    # 0 = Building OR (09:30-09:45 NY -> min_of_day 570, 575, 580)
    # 1 = Range Locked -> Waiting for Breakout Close (09:45-11:55 NY -> min_of_day 585..710)
    # 2 = Trade 1 Limit Pending (09:45-12:00 NY -> min_of_day 585..715)
    # 3 = Trade 1 Open (Runs until SL/TP or next day 09:30 reset)
    # 4 = Trade 1 Stopped Out by Wick -> Waiting for Candle Close Beyond OR for Trade 2 (up to 12:00 close)
    # 5 = Trade 2 Open (Runs until SL/TP or next day 09:30 reset)
    # 6 = Session Done
    state = 6
    cur_day = -1

    orh = 0.0
    orl = 0.0
    orw = 0.0
    or_valid = False

    t1_dir = 0
    t1_entry = 0.0
    t1_sl = 0.0
    t1_tp = 0.0
    t1_risk = 0.0
    t1_qty = 0.0
    t1_fill_idx = 0
    recent_ext = 0.0

    t2_dir = 0
    t2_entry = 0.0
    t2_sl = 0.0
    t2_tp = 0.0
    t2_risk = 0.0
    t2_qty = 0.0
    t2_fill_idx = 0

    for i in range(n):
        o = open_arr[i]
        h = high_arr[i]
        l = low_arr[i]
        c = close_arr[i]
        d_id = day_id_arr[i]
        m_od = min_of_day_arr[i]

        # ------------------------------------------------------------------
        # 1. DETECT START OF 09:30 NY OPENING RANGE (570 mins from midnight)
        # ------------------------------------------------------------------
        if m_od >= 570 and m_od < 585:
            if state != 0 or d_id != cur_day:
                # Force-close any leftover overnight trade at 09:30 open
                if state == 3:
                    gross = (o - t1_entry) * t1_dir * t1_qty * point_value
                    net   = gross - (comm_per_unit * t1_qty)
                    tr_entry_idx[t_count] = t1_fill_idx
                    tr_exit_idx[t_count]  = i
                    tr_leg[t_count]       = 1
                    tr_dir[t_count]       = t1_dir
                    tr_qty[t_count]       = t1_qty
                    tr_entry_px[t_count]  = t1_entry
                    tr_exit_px[t_count]   = o
                    tr_sl_px[t_count]     = t1_sl
                    tr_tp_px[t_count]     = t1_tp
                    tr_pnl_usd[t_count]   = net
                    tr_r_mult[t_count]    = ((o - t1_entry) * t1_dir) / t1_risk
                    t_count += 1
                elif state == 5:
                    gross = (o - t2_entry) * t2_dir * t2_qty * point_value
                    net   = gross - (comm_per_unit * t2_qty)
                    tr_entry_idx[t_count] = t2_fill_idx
                    tr_exit_idx[t_count]  = i
                    tr_leg[t_count]       = 2
                    tr_dir[t_count]       = t2_dir
                    tr_qty[t_count]       = t2_qty
                    tr_entry_px[t_count]  = t2_entry
                    tr_exit_px[t_count]   = o
                    tr_sl_px[t_count]     = t2_sl
                    tr_tp_px[t_count]     = t2_tp
                    tr_pnl_usd[t_count]   = net
                    tr_r_mult[t_count]    = ((o - t2_entry) * t2_dir) / t2_risk
                    t_count += 1

                # Reset for new Opening Range
                cur_day  = d_id
                state    = 0
                orh      = h
                orl      = l
                or_valid = False
            else:
                if h > orh:
                    orh = h
                if l < orl:
                    orl = l
            if orh > orl:
                or_valid = True
            continue

        # Transition from State 0 -> State 1 at 09:45 NY (min_of_day == 585)
        if state == 0 and m_od >= 585:
            if or_valid:
                orw   = orh - orl
                state = 1
            else:
                state = 6

        # Helper flags for Active Trading Window (09:45 - 12:00 NY)
        # Note: 5m bar with m_od == 715 opens at 11:55 and closes at 12:00:00 NY
        in_trade_window    = (d_id == cur_day) and (m_od >= 585) and (m_od <= 715)
        cutoff_at_bar_close = (m_od >= 715)

        # ------------------------------------------------------------------
        # 2. STATE 1: WAITING FOR INITIAL BREAKOUT CLOSE OUTSIDE [ORL, ORH]
        # ------------------------------------------------------------------
        if state == 1:
            if not in_trade_window or cutoff_at_bar_close:
                state = 6
            else:
                if c > orh:
                    t1_dir     = 1
                    t1_entry   = round_tick(orh - (fib_lvl * orw), tick_size)
                    t1_sl      = round_tick(orl, tick_size)
                    t1_risk    = max(t1_entry - t1_sl, tick_size)
                    recent_ext = h
                    raw_dist   = recent_ext - t1_entry
                    clamp_dist = min(t1_max_rr * t1_risk, max(t1_min_rr * t1_risk, raw_dist))
                    t1_tp      = round_tick(t1_entry + clamp_dist, tick_size)
                    t1_qty     = calc_position_size(t1_risk, risk_usd, point_value, tick_size, is_futures)
                    state      = 2
                elif c < orl:
                    t1_dir     = -1
                    t1_entry   = round_tick(orl + (fib_lvl * orw), tick_size)
                    t1_sl      = round_tick(orh, tick_size)
                    t1_risk    = max(t1_sl - t1_entry, tick_size)
                    recent_ext = l
                    raw_dist   = t1_entry - recent_ext
                    clamp_dist = min(t1_max_rr * t1_risk, max(t1_min_rr * t1_risk, raw_dist))
                    t1_tp      = round_tick(t1_entry - clamp_dist, tick_size)
                    t1_qty     = calc_position_size(t1_risk, risk_usd, point_value, tick_size, is_futures)
                    state      = 2

        # ------------------------------------------------------------------
        # 3. STATE 2: TRADE 1 PENDING LIMIT ORDER (0.618 PULLBACK)
        # ------------------------------------------------------------------
        elif state == 2:
            if not in_trade_window:
                state = 6
            else:
                filled = (l <= t1_entry) if t1_dir == 1 else (h >= t1_entry)
                if filled:
                    t1_fill_idx = i
                    # Check same-bar SL first (Option 3A: Conservative SL priority)
                    hit_sl = (l <= t1_sl) if t1_dir == 1 else (h >= t1_sl)
                    if hit_sl:
                        gross = (t1_sl - t1_entry) * t1_dir * t1_qty * point_value
                        net   = gross - (comm_per_unit * t1_qty)
                        tr_entry_idx[t_count] = t1_fill_idx
                        tr_exit_idx[t_count]  = i
                        tr_leg[t_count]       = 1
                        tr_dir[t_count]       = t1_dir
                        tr_qty[t_count]       = t1_qty
                        tr_entry_px[t_count]  = t1_entry
                        tr_exit_px[t_count]   = t1_sl
                        tr_sl_px[t_count]     = t1_sl
                        tr_tp_px[t_count]     = t1_tp
                        tr_pnl_usd[t_count]   = net
                        tr_r_mult[t_count]    = -1.0
                        t_count += 1

                        # Check if this same bar closed beyond ORL/ORH to trigger Trade 2 immediately
                        rev_close = (c < orl) if t1_dir == 1 else (c > orh)
                        if rev_close:
                            t2_dir      = -t1_dir
                            t2_entry    = round_tick(c, tick_size)
                            t2_sl       = round_tick(h if t2_dir == -1 else l, tick_size)
                            t2_risk     = max((t2_sl - t2_entry) if t2_dir == -1 else (t2_entry - t2_sl), tick_size)
                            t2_tp       = round_tick(t2_entry + (t2_dir * t2_fix_rr * t2_risk), tick_size)
                            t2_qty      = calc_position_size(t2_risk, risk_usd, point_value, tick_size, is_futures)
                            t2_fill_idx = i
                            state       = 5
                        elif not cutoff_at_bar_close:
                            state = 4
                        else:
                            state = 6
                    else:
                        # Lock TP once filled; transition to active Trade 1
                        state = 3
                else:
                    # Unfilled: Cancel if 12:00 NY cutoff reached, else update dynamic peak TP
                    if cutoff_at_bar_close:
                        state = 6
                    else:
                        if t1_dir == 1:
                            if h > recent_ext:
                                recent_ext = h
                            raw_dist   = recent_ext - t1_entry
                            clamp_dist = min(t1_max_rr * t1_risk, max(t1_min_rr * t1_risk, raw_dist))
                            t1_tp      = round_tick(t1_entry + clamp_dist, tick_size)
                        else:
                            if l < recent_ext:
                                recent_ext = l
                            raw_dist   = t1_entry - recent_ext
                            clamp_dist = min(t1_max_rr * t1_risk, max(t1_min_rr * t1_risk, raw_dist))
                            t1_tp      = round_tick(t1_entry - clamp_dist, tick_size)

        # ------------------------------------------------------------------
        # 4. STATE 3: TRADE 1 ACTIVE POSITION
        # ------------------------------------------------------------------
        elif state == 3:
            # Option 3A: Check SL first, then TP
            hit_sl = (l <= t1_sl) if t1_dir == 1 else (h >= t1_sl)
            hit_tp = (h >= t1_tp) if t1_dir == 1 else (l <= t1_tp)

            if hit_sl:
                gross = (t1_sl - t1_entry) * t1_dir * t1_qty * point_value
                net   = gross - (comm_per_unit * t1_qty)
                tr_entry_idx[t_count] = t1_fill_idx
                tr_exit_idx[t_count]  = i
                tr_leg[t_count]       = 1
                tr_dir[t_count]       = t1_dir
                tr_qty[t_count]       = t1_qty
                tr_entry_px[t_count]  = t1_entry
                tr_exit_px[t_count]   = t1_sl
                tr_sl_px[t_count]     = t1_sl
                tr_tp_px[t_count]     = t1_tp
                tr_pnl_usd[t_count]   = net
                tr_r_mult[t_count]    = -1.0
                t_count += 1

                rev_close = (c < orl) if t1_dir == 1 else (c > orh)
                if in_trade_window and rev_close:
                    t2_dir      = -t1_dir
                    t2_entry    = round_tick(c, tick_size)
                    t2_sl       = round_tick(h if t2_dir == -1 else l, tick_size)
                    t2_risk     = max((t2_sl - t2_entry) if t2_dir == -1 else (t2_entry - t2_sl), tick_size)
                    t2_tp       = round_tick(t2_entry + (t2_dir * t2_fix_rr * t2_risk), tick_size)
                    t2_qty      = calc_position_size(t2_risk, risk_usd, point_value, tick_size, is_futures)
                    t2_fill_idx = i
                    state       = 5
                elif in_trade_window and not cutoff_at_bar_close:
                    state = 4
                else:
                    state = 6

            elif hit_tp:
                gross = (t1_tp - t1_entry) * t1_dir * t1_qty * point_value
                net   = gross - (comm_per_unit * t1_qty)
                tr_entry_idx[t_count] = t1_fill_idx
                tr_exit_idx[t_count]  = i
                tr_leg[t_count]       = 1
                tr_dir[t_count]       = t1_dir
                tr_qty[t_count]       = t1_qty
                tr_entry_px[t_count]  = t1_entry
                tr_exit_px[t_count]   = t1_tp
                tr_sl_px[t_count]     = t1_sl
                tr_tp_px[t_count]     = t1_tp
                tr_pnl_usd[t_count]   = net
                tr_r_mult[t_count]    = ((t1_tp - t1_entry) * t1_dir) / t1_risk
                t_count += 1
                state = 6

        # ------------------------------------------------------------------
        # 5. STATE 4: T1 STOPPED BY WICK -> WAITING FOR BREAKOUT CLOSE FOR T2
        # ------------------------------------------------------------------
        elif state == 4:
            if not in_trade_window:
                state = 6
            else:
                rev_close = (c < orl) if t1_dir == 1 else (c > orh)
                if rev_close:
                    t2_dir      = -t1_dir
                    t2_entry    = round_tick(c, tick_size)
                    t2_sl       = round_tick(h if t2_dir == -1 else l, tick_size)
                    t2_risk     = max((t2_sl - t2_entry) if t2_dir == -1 else (t2_entry - t2_sl), tick_size)
                    t2_tp       = round_tick(t2_entry + (t2_dir * t2_fix_rr * t2_risk), tick_size)
                    t2_qty      = calc_position_size(t2_risk, risk_usd, point_value, tick_size, is_futures)
                    t2_fill_idx = i
                    state       = 5
                elif cutoff_at_bar_close:
                    state = 6

        # ------------------------------------------------------------------
        # 6. STATE 5: TRADE 2 ACTIVE POSITION (FIXED 2R REVERSAL)
        # ------------------------------------------------------------------
        elif state == 5:
            hit_sl = (l <= t2_sl) if t2_dir == 1 else (h >= t2_sl)
            hit_tp = (h >= t2_tp) if t2_dir == 1 else (l <= t2_tp)

            if hit_sl:
                gross = (t2_sl - t2_entry) * t2_dir * t2_qty * point_value
                net   = gross - (comm_per_unit * t2_qty)
                tr_entry_idx[t_count] = t2_fill_idx
                tr_exit_idx[t_count]  = i
                tr_leg[t_count]       = 2
                tr_dir[t_count]       = t2_dir
                tr_qty[t_count]       = t2_qty
                tr_entry_px[t_count]  = t2_entry
                tr_exit_px[t_count]   = t2_sl
                tr_sl_px[t_count]     = t2_sl
                tr_tp_px[t_count]     = t2_tp
                tr_pnl_usd[t_count]   = net
                tr_r_mult[t_count]    = -1.0
                t_count += 1
                state = 6

            elif hit_tp:
                gross = (t2_tp - t2_entry) * t2_dir * t2_qty * point_value
                net   = gross - (comm_per_unit * t2_qty)
                tr_entry_idx[t_count] = t2_fill_idx
                tr_exit_idx[t_count]  = i
                tr_leg[t_count]       = 2
                tr_dir[t_count]       = t2_dir
                tr_qty[t_count]       = t2_qty
                tr_entry_px[t_count]  = t2_entry
                tr_exit_px[t_count]   = t2_tp
                tr_sl_px[t_count]     = t2_sl
                tr_tp_px[t_count]     = t2_tp
                tr_pnl_usd[t_count]   = net
                tr_r_mult[t_count]    = t2_fix_rr
                t_count += 1
                state = 6

    return (
        tr_entry_idx[:t_count],
        tr_exit_idx[:t_count],
        tr_leg[:t_count],
        tr_dir[:t_count],
        tr_qty[:t_count],
        tr_entry_px[:t_count],
        tr_exit_px[:t_count],
        tr_sl_px[:t_count],
        tr_tp_px[:t_count],
        tr_pnl_usd[:t_count],
        tr_r_mult[:t_count]
    )

## Cell 3: Execute Backtest, Leg Attribution & Prop Firm Survival Audit

In [10]:
# ==============================================================================
# CELL 3 (FIXED): 0-BYTE SAFE BACKTEST EXECUTION + TEARSHEET + PROP AUDIT
# ==============================================================================
import os
import glob
import numpy as np
import pandas as pd
import yfinance as yf

# --- 1. VERIFY df_5m IS LOADED & NON-EMPTY ---
if "df_5m" not in globals() or df_5m is None or len(df_5m) == 0:
    print("[AUTO-FIX] 'df_5m' not in memory. Fetching 60d 5m MNQ=F via Yahoo Finance...")
    for f in glob.glob("./data/*.csv"):
        if os.path.getsize(f) < 500:
            os.remove(f)
    raw_df = yf.download("MNQ=F", period="60d", interval="5m", progress=False)
    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df.columns = raw_df.columns.get_level_values(0)
    raw_df.index = pd.to_datetime(raw_df.index, utc=True)
    raw_df.columns = [str(c).lower() for c in raw_df.columns]
    df_5m = raw_df[["open", "high", "low", "close"]].dropna()
    df_5m = df_5m[~df_5m.index.duplicated(keep="first")]
    df_5m.index = df_5m.index.tz_convert("America/New_York")

if "rorb_state_machine_kernel" not in globals():
    raise RuntimeError("Please run Cell 2 first to compile the Numba state machine!")

# --- 2. SIMULATION & PROP FIRM SETTINGS ---
SIM_MODE        = "MNQ_FUTURES"  # Options: "MNQ_FUTURES" or "NAS100_CFD"
INITIAL_CAPITAL = 50000.0
RISK_PER_TRADE  = 300.0
PROP_PROFIT_TGT = 3000.0         # $50k Prop Eval Target ($3,000)
PROP_MAX_DD     = 2000.0         # $50k Prop Trailing Drawdown ($2,000)

if SIM_MODE == "MNQ_FUTURES":
    POINT_VALUE   = 2.0          # MNQ = $2.00 per index point
    TICK_SIZE     = 0.25         # CME MNQ minimum tick = 0.25 pts
    COMM_PER_UNIT = 1.40         # $1.40 round-turn commission per contract
    IS_FUTURES    = True
else:
    POINT_VALUE   = 1.0          # CFD = $1.00 per point per unit
    TICK_SIZE     = 0.10
    COMM_PER_UNIT = 0.0
    IS_FUTURES    = False

# --- 3. RUN BACKTEST ---
def run_rorb_backtest(df: pd.DataFrame) -> pd.DataFrame:
    open_arr  = df["open"].to_numpy(dtype=np.float64)
    high_arr  = df["high"].to_numpy(dtype=np.float64)
    low_arr   = df["low"].to_numpy(dtype=np.float64)
    close_arr = df["close"].to_numpy(dtype=np.float64)

    ny_dates       = df.index.year * 10000 + df.index.month * 100 + df.index.day
    day_id_arr     = ny_dates.to_numpy(dtype=np.int64)
    min_of_day_arr = (df.index.hour * 60 + df.index.minute).to_numpy(dtype=np.int64)

    res = rorb_state_machine_kernel(
        open_arr, high_arr, low_arr, close_arr,
        day_id_arr, min_of_day_arr,
        risk_usd=RISK_PER_TRADE,
        point_value=POINT_VALUE,
        tick_size=TICK_SIZE,
        comm_per_unit=COMM_PER_UNIT,
        is_futures=IS_FUTURES,
        fib_lvl=0.618,
        t1_min_rr=1.0,
        t1_max_rr=3.0,
        t2_fix_rr=2.0
    )

    trades = pd.DataFrame({
        "entry_time": df.index[res[0]],
        "exit_time":  df.index[res[1]],
        "leg":        np.where(res[2] == 1, "Trade 1 (0.618 Limit)", "Trade 2 (2R Reversal)"),
        "leg_num":    res[2],
        "direction":  np.where(res[3] == 1, "LONG", "SHORT"),
        "qty":        res[4],
        "entry_px":   res[5],
        "exit_px":    res[6],
        "sl_px":      res[7],
        "tp_px":      res[8],
        "pnl_usd":    res[9],
        "r_multiple": res[10]
    })
    trades["trade_date"] = trades["entry_time"].dt.date
    trades["cum_pnl"]    = trades["pnl_usd"].cumsum()
    trades["equity"]     = INITIAL_CAPITAL + trades["cum_pnl"]
    trades["peak_eq"]    = np.maximum(INITIAL_CAPITAL, trades["equity"].cummax())
    trades["dd_usd"]     = trades["equity"] - trades["peak_eq"]
    return trades

trades_df = run_rorb_backtest(df_5m)

# --- 4. PERFORMANCE TEARSHEET ---
def compute_leg_stats(sub_df: pd.DataFrame, label: str) -> dict:
    if len(sub_df) == 0:
        return {"Segment": label, "Trades": 0}
    wins       = sub_df[sub_df["pnl_usd"] > 0]
    losses     = sub_df[sub_df["pnl_usd"] <= 0]
    win_rate   = len(wins) / len(sub_df) * 100.0
    gross_win  = wins["pnl_usd"].sum()
    gross_loss = abs(losses["pnl_usd"].sum())
    pf         = gross_win / gross_loss if gross_loss > 0 else np.inf
    net_pnl    = sub_df["pnl_usd"].sum()
    exp_usd    = sub_df["pnl_usd"].mean()
    exp_r      = sub_df["r_multiple"].mean()

    daily_pnl  = sub_df.groupby("trade_date")["pnl_usd"].sum()
    sharpe     = (daily_pnl.mean() / daily_pnl.std()) * np.sqrt(252) if len(daily_pnl) > 1 and daily_pnl.std() > 0 else 0.0
    downside   = daily_pnl[daily_pnl < 0]
    sortino    = (daily_pnl.mean() / downside.std()) * np.sqrt(252) if len(downside) > 1 and downside.std() > 0 else 0.0

    eq         = INITIAL_CAPITAL + sub_df["pnl_usd"].cumsum()
    peak       = np.maximum(INITIAL_CAPITAL, eq.cummax())
    max_dd_usd = (eq - peak).min()

    return {
        "Segment":          label,
        "Trades":           len(sub_df),
        "Win Rate (%)":     round(win_rate, 2),
        "Profit Factor":    round(pf, 2),
        "Net PnL ($)":      round(net_pnl, 2),
        "Return (%)":       round((net_pnl / INITIAL_CAPITAL) * 100, 2),
        "Expectancy ($)":   round(exp_usd, 2),
        "Expectancy (R)":   round(exp_r, 3),
        "Max DD ($)":       round(max_dd_usd, 2),
        "Sharpe (Ann.)":    round(sharpe, 2),
        "Sortino (Ann.)":   round(sortino, 2)
    }

summary_table = pd.DataFrame([
    compute_leg_stats(trades_df, "ALL TRADES (Combined RORB)"),
    compute_leg_stats(trades_df[trades_df["leg_num"] == 1], "Trade 1 Only (0.618 Pullback)"),
    compute_leg_stats(trades_df[trades_df["leg_num"] == 2], "Trade 2 Only (2R Failure Reversal)")
])

print("=" * 95)
print(f" RORB STRATEGY TEARSHEET | MODE: {SIM_MODE} | RISK: ${RISK_PER_TRADE:.0f}/Trade")
print("=" * 95)
display(summary_table.set_index("Segment"))

# --- 5. PROP FIRM SURVIVAL AUDIT ---
def prop_firm_audit(df_tr: pd.DataFrame, target_usd: float, max_dd_limit: float):
    eval_equity = 0.0
    eval_peak   = 0.0
    passes      = 0
    breaches    = 0

    for pnl in df_tr["pnl_usd"]:
        eval_equity += pnl
        if eval_equity > eval_peak:
            eval_peak = eval_equity
        if eval_equity >= target_usd:
            passes += 1
            eval_equity = 0.0
            eval_peak   = 0.0
        elif (eval_peak - eval_equity) >= max_dd_limit:
            breaches += 1
            eval_equity = 0.0
            eval_peak   = 0.0

    daily_pnl      = df_tr.groupby("trade_date")["pnl_usd"].sum()
    worst_day      = daily_pnl.min() if len(daily_pnl) > 0 else 0.0
    max_hist_dd    = abs(df_tr["dd_usd"].min()) if len(df_tr) > 0 else 0.0
    safe_prop_risk = RISK_PER_TRADE * ((max_dd_limit * 0.70) / max_hist_dd) if max_hist_dd > 0 else RISK_PER_TRADE

    print("\n" + "=" * 95)
    print(f" PROP FIRM SURVIVAL AUDIT ($50k Account | +${target_usd:,.0f} Target vs -${max_dd_limit:,.0f} Trailing DD)")
    print("=" * 95)
    print(f" • Historical Max Peak-to-Valley Drawdown : -${max_hist_dd:,.2f}")
    print(f" • Worst Single Day PnL                   : ${worst_day:,.2f}")
    print(f" • Sequential Prop Evaluations Passed     : {passes} Passes vs. {breaches} Breaches")
    print(f" • Recommended Safe Risk/Trade for Prop   : ${safe_prop_risk:,.2f} (keeps historical Max DD at 70% of -${max_dd_limit:,.0f} limit)")
    print("=" * 95)

prop_firm_audit(trades_df, PROP_PROFIT_TGT, PROP_MAX_DD)

 RORB STRATEGY TEARSHEET | MODE: MNQ_FUTURES | RISK: $300/Trade


,Trades,Win Rate (%),Profit Factor,Net PnL ($),Return (%),Expectancy ($),Expectancy (R),Max DD ($),Sharpe (Ann.),Sortino (Ann.)
Segment,,,,,,,,,,
ALL TRADES (Combined RORB),35,31.43,1.02,118.6,0.24,3.39,-0.027,-2373.0,0.15,0.49
Trade 1 Only (0.618 Pullback),23,26.09,0.78,-1058.3,-2.12,-46.01,-0.172,-2010.8,-1.74,-23.95
Trade 2 Only (2R Failure Reversal),12,41.67,1.53,1176.9,2.35,98.08,0.250,-1284.4,3.04,45.06



 PROP FIRM SURVIVAL AUDIT ($50k Account | +$3,000 Target vs -$2,000 Trailing DD)
 • Historical Max Peak-to-Valley Drawdown : -$2,373.00
 • Worst Single Day PnL                   : $-636.60
 • Sequential Prop Evaluations Passed     : 0 Passes vs. 1 Breaches
 • Recommended Safe Risk/Trade for Prop   : $176.99 (keeps historical Max DD at 70% of -$2,000 limit)


### Cell 4: Interactive Plotly Equity Curve, Underwater Drawdown & Monthly PnL

In [11]:
# ==============================================================================
# CELL 4 (FIXED): INTERACTIVE PLOTLY DASHBOARD (PANDAS VERSION-SAFE)
# ==============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if "trades_df" not in globals() or trades_df is None:
    raise RuntimeError("Please run Cell 3 above first so 'trades_df' is generated!")

if len(trades_df) > 0:
    # Build separate cumulative PnL curves for Leg 1 and Leg 2
    t_idx = trades_df.set_index("exit_time")
    t1_series = t_idx["pnl_usd"].where(t_idx["leg_num"] == 1, 0.0).cumsum()
    t2_series = t_idx["pnl_usd"].where(t_idx["leg_num"] == 2, 0.0).cumsum()

    # Version-safe Monthly PnL Aggregation (works across all Pandas versions)
    month_labels = trades_df["exit_time"].dt.strftime("%Y-%m")
    monthly_pnl  = trades_df.groupby(month_labels)["pnl_usd"].sum()
    month_colors = ["#089981" if v >= 0 else "#f23645" for v in monthly_pnl.values]

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.08,
        row_heights=[0.45, 0.30, 0.25],
        subplot_titles=(
            f"Cumulative PnL ($) — Combined vs. Trade 1 (0.618 Limit) vs. Trade 2 (2R Reversal) [{SIM_MODE}]",
            "Underwater Drawdown ($) vs. $50k Prop Firm Trailing Limits (-$2,000 & -$2,500)",
            "Monthly Net Profit ($)"
        )
    )

    # Panel 1: Cumulative PnL Curves
    fig.add_trace(go.Scatter(
        x=trades_df["exit_time"], y=trades_df["cum_pnl"],
        mode="lines", name="Combined RORB PnL ($)",
        line=dict(color="#00E676", width=2.5)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=t1_series.index, y=t1_series.values,
        mode="lines", name="Trade 1 Only (0.618 Limit)",
        line=dict(color="#29B6F6", width=1.5, dash="dot")
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=t2_series.index, y=t2_series.values,
        mode="lines", name="Trade 2 Only (2R Reversal)",
        line=dict(color="#FFA726", width=1.5, dash="dash")
    ), row=1, col=1)

    # Panel 2: Underwater Dollar Drawdown + Prop Firm Breach Lines
    fig.add_trace(go.Scatter(
        x=trades_df["exit_time"], y=trades_df["dd_usd"],
        mode="lines", fill="tozeroy", name="Drawdown ($)",
        line=dict(color="#f23645", width=1.5),
        fillcolor="rgba(242, 54, 69, 0.25)"
    ), row=2, col=1)

    fig.add_hline(y=-2000, line_dash="dash", line_color="#FFEB3B",
                  annotation_text="-$2,000 Prop Limit (Lucid/Topstep/Bulenox)", row=2, col=1)
    fig.add_hline(y=-2500, line_dash="dot", line_color="#FF5252",
                  annotation_text="-$2,500 Prop Limit (Apex 50k)", row=2, col=1)

    # Panel 3: Monthly PnL Bars
    fig.add_trace(go.Bar(
        x=monthly_pnl.index, y=monthly_pnl.values,
        name="Monthly PnL ($)", marker_color=month_colors
    ), row=3, col=1)

    fig.update_layout(
        template="plotly_dark",
        height=950,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig.show()
else:
    print("[WARN] No trades were triggered in the dataset.")